In [ ]:
# SECTION 1 — Imports and configuration
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, precision_score, recall_score

import warnings
warnings.filterwarnings('ignore')

# REPRODUCIBILITY
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# KAGGLE SAFETY: Print device info
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("CUDA availability:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


In [ ]:
# SECTION 2 — Dataset discovery and validation
base_path = "/kaggle/input/datasets/manan1717/ocular-disease-dataset/preprocessed"

total = 0
bad = []

if os.path.exists(base_path):
    print("dataset path:", base_path)
    for cls in sorted(os.listdir(base_path)):
        folder = os.path.join(base_path, cls)
        if not os.path.isdir(folder):
            continue
        for filename in os.listdir(folder):
            path = os.path.join(folder, filename)
            total += 1
            try:
                with Image.open(path) as img:
                    img.verify()
            except Exception as e:
                bad.append((filename, str(e)))

    print("number of images:", total)
    print("Bad/unreadable images:", len(bad))
else:
    print(f"Dataset path {base_path} not found. Please ensure it's attached.")


In [ ]:
# SECTION 3 — Class mapping
class_to_idx = {
    "N": 0,
    "D": 1,
    "G": 2,
    "C": 3,
    "A": 4,
    "H": 5,
    "M": 6
}
idx_to_class = {v: k for k, v in class_to_idx.items()}
classes = ["N", "D", "G", "C", "A", "H", "M"]

data = []
if os.path.exists(base_path):
    for label in classes:
        folder = os.path.join(base_path, label)
        if os.path.isdir(folder):
            for filename in os.listdir(folder):
                data.append({
                    "filename": filename,
                    "label": label,
                    "path": os.path.join(folder, filename)
                })

df = pd.DataFrame(data)
print("\nclass distribution:")
if len(df) > 0:
    print(df["label"].value_counts()[classes])


In [ ]:
# SECTION 4 — Stratified train/validation/test split
# Previous split: Train = 8359, Validation = 1045, Test = 1045
# Total images = 10449
# 80% Train, 10% Validation, 10% Test

if len(df) > 0:
    train_df, temp_df = train_test_split(
        df,
        test_size=0.20,
        stratify=df["label"],
        random_state=42
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df["label"],
        random_state=42
    )
    
    # Reset indices
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print("train/validation/test counts:")
    print("Train:", len(train_df))
    print("Validation:", len(val_df))
    print("Test:", len(test_df))
    
    # Verify no overlap
    train_paths = set(train_df['path'])
    val_paths = set(val_df['path'])
    test_paths = set(test_df['path'])
    
    print("Train/Val overlap:", len(train_paths.intersection(val_paths)))
    print("Train/Test overlap:", len(train_paths.intersection(test_paths)))
    print("Val/Test overlap:", len(val_paths.intersection(test_paths)))


In [ ]:
# SECTION 5 — Image transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
# SECTION 6 — Dataset and DataLoaders
class OcularDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        label = class_to_idx[row["label"]]

        if self.transform:
            image = self.transform(image)

        return image, label

if len(df) > 0:
    train_dataset = OcularDataset(train_df, train_transform)
    val_dataset = OcularDataset(val_df, eval_transform)
    test_dataset = OcularDataset(test_df, eval_transform)

    batch_size = 64

    # Kaggle num_workers issue -> use 0
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    print("DataLoaders created.")


In [ ]:
# SECTION 7 — ResNet-50 Model

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Load pretrained ResNet-50
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)

# Replace final layer for 7 classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(classes))

model = model.to(device)

print("ResNet-50 loaded successfully.")
print("Number of classes:", len(classes))

In [ ]:
# SECTION 8 — Loss, Optimizer and Mixed Precision

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else torch.amp.GradScaler("cpu")

print("Training setup ready.")

In [ ]:
# SECTION 8.5 — Training and Evaluation Functions

from sklearn.metrics import f1_score
from tqdm.auto import tqdm

def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if device.type == "cuda":
            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_labels = []
    all_predictions = []

    for images, labels in tqdm(loader, desc="Validation", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if device.type == "cuda":
            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())

    total = len(all_labels)

    val_loss = running_loss / total
    val_acc = sum(
        p == y for p, y in zip(all_predictions, all_labels)
    ) / total

    val_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )

    return val_loss, val_acc, val_f1, all_labels, all_predictions


print("Training and evaluation functions ready.")

In [ ]:
# SECTION 9 — Training loop with checkpoint saving
if len(df) > 0:
    epochs = 10
    best_val_f1 = 0.0
    best_epoch = -1
    
    history = {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "val_f1": [],
        "lr": []
    }

    checkpoint_path = "/kaggle/working/resnet50_best.pth"
    history_path = "/kaggle/working/resnet50_training_history.csv"
    
    # Ensure working directory exists for testing outside kaggle if needed
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler, device)
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        
        current_lr = optimizer.param_groups[0]['lr']
        
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)
        history["lr"].append(current_lr)
        
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val Macro F1: {val_f1:.4f}")
        
        # Checkpoint saving
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            print(f"⭐ New best validation Macro F1: {best_val_f1:.4f}. Saving checkpoint...")
            
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_macro_f1': best_val_f1,
                'class_names': classes
            }
            torch.save(checkpoint, checkpoint_path)
            
        # Save history
        pd.DataFrame(history).to_csv(history_path, index=False)


In [ ]:
import os

print("Files in /kaggle/working:")
print(os.listdir("/kaggle/working"))

print("\nViT checkpoint exists:",
      os.path.exists("/kaggle/working/vit_b16_best.pth"))

print("History exists:",
      os.path.exists("/kaggle/working/vit_b16_training_history.csv"))

In [ ]:
import os

print("ResNet checkpoint exists:",
      os.path.exists("/kaggle/working/resnet50_best.pth"))

print("ResNet history exists:",
      os.path.exists("/kaggle/working/resnet50_training_history.csv"))

In [ ]:
# SECTION 10 — Restore best checkpoint
if len(df) > 0:
    if os.path.exists(checkpoint_path):
        print(f"Loading best checkpoint from epoch {best_epoch}...")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded checkpoint with Validation Macro F1: {checkpoint['best_val_macro_f1']:.4f}")
    else:
        print("Checkpoint not found!")


In [ ]:
# SECTION 11 — Test evaluation
if len(df) > 0:
    print("Evaluating on test set...")
    test_loss, test_acc, test_macro_f1, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

    test_macro_prec = precision_score(test_labels, test_preds, average="macro")
    test_macro_rec = recall_score(test_labels, test_preds, average="macro")
    test_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")
    
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro Precision: {test_macro_prec:.4f}")
    print(f"Test Macro Recall: {test_macro_rec:.4f}")
    print(f"Test Macro F1: {test_macro_f1:.4f}")
    print(f"Test Weighted F1: {test_weighted_f1:.4f}")


In [ ]:
# SECTION 12 — Classification report
if len(df) > 0:
    print("\nClassification Report:")
    print(classification_report(test_labels, test_preds, target_names=classes, digits=4))


In [ ]:
# SECTION 13 — Confusion matrix
if len(df) > 0:
    cm = confusion_matrix(test_labels, test_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title("ResNet-50 Test Confusion Matrix")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()


In [ ]:
# SECTION 14 — Training curves
if len(df) > 0 and os.path.exists(history_path):
    hist_df = pd.read_csv(history_path)
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', marker='o')
    plt.plot(hist_df['epoch'], hist_df['val_loss'], label='Val Loss', marker='o')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(hist_df['epoch'], hist_df['train_acc'], label='Train Acc', marker='o')
    plt.plot(hist_df['epoch'], hist_df['val_acc'], label='Val Acc', marker='o')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(6, 5))
    plt.plot(hist_df['epoch'], hist_df['val_f1'], label='Val Macro F1', marker='o', color='green')
    plt.title('Validation Macro F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('Macro F1')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# SECTION 15 — Per-class F1
if len(df) > 0:
    class_f1 = f1_score(test_labels, test_preds, average=None)
    
    plt.figure(figsize=(8, 5))
    sns.barplot(x=classes, y=class_f1, palette='viridis')
    plt.title('Per-Class F1 Score on Test Set')
    plt.xlabel('Class')
    plt.ylabel('F1 Score')
    plt.ylim(0, 1)
    for i, v in enumerate(class_f1):
        plt.text(i, v + 0.02, f"{v:.4f}", ha='center', va='bottom', fontsize=9)
    plt.show()


In [ ]:
# SECTION 16 — External image prediction
def predict_image(img_path, model, transform, device):
    if not os.path.exists(img_path):
        return None
    image = Image.open(img_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            outputs = model(image_tensor)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            pred_idx = probs.argmax(dim=1).item()
            pred_prob = probs[0, pred_idx].item()
    return classes[pred_idx], pred_prob


In [ ]:
# SECTION 17 — Final checkpoint verification
if len(df) > 0:
    print("\nRESNET-50 EXPERIMENT COMPLETE")
    print(f"Best validation Macro F1: {best_val_f1:.4f}")
    print(f"Best epoch: {best_epoch}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro F1: {test_macro_f1:.4f}")
    print(f"Test Precision: {test_macro_prec:.4f}")
    print(f"Test Recall: {test_macro_rec:.4f}")
    if os.path.exists(checkpoint_path):
        print(f"Checkpoint path: {checkpoint_path}")
        print(f"Checkpoint file size: {os.path.getsize(checkpoint_path) / (1024*1024):.2f} MB")
    if os.path.exists(history_path):
        print(f"Training history path: {history_path}")
